### Testing how to use my proposed equation

In [ ]:
import numpy as np
import pandas as pd

# import and concatenate shear stress data
shear_stress_2022 = pd.read_csv("field_data/shear_stress_2022.csv", parse_dates=["time"])
shear_stress_2023 = pd.read_csv("field_data/shear_stress_2023.csv", parse_dates=["time"])
shear_stress = pd.concat([shear_stress_2022, shear_stress_2023])
# sort by time just to be safe
shear_stress = shear_stress.sort_values("time")

def antecedent_average(
    events_csv,
    shear_stress,
    event_time_col="date",
    shear_time_col="time",
    shear_col="tau*"
    ):
    # read data
    events = pd.read_csv(events_csv, parse_dates=[event_time_col]) 
    events = events.sort_values(event_time_col).reset_index(drop=True) # sort just to be safe
    results = []

    for i in range(len(events) - 1):
        t_start = events.loc[i, event_time_col]
        t_end = events.loc[i + 1, event_time_col]
        # subset shear stress between events
        mask = (shear_stress[shear_time_col] > t_start) & (shear_stress[shear_time_col] < t_end)
        shear_between = shear_stress.loc[mask, shear_col]
        # time between events (in hours)
        delta_t_hours = (t_end - t_start).total_seconds() / 3600
        # mean shear stress between events
        mean_tau = shear_between.mean()

        results.append({
            "event_start": t_start,
            "event_end": t_end,
            "time_between_events_hours": delta_t_hours,
            "mean_shear_between_events": mean_tau,
            "n_timesteps": shear_between.size
        })
    return pd.DataFrame(results)

import pandas as pd

def antecedent_average_below_tauc(
    events_csv,
    shear_stress,
    event_time_col="date",
    tauc_col="tauc*",
    shear_time_col="time",
    shear_col="tau*"
):
    # read and sort events
    events = pd.read_csv(events_csv, parse_dates=[event_time_col])
    events = events.sort_values(event_time_col).reset_index(drop=True)

    results = []

    for i in range(len(events) - 1):
        t_start = events.loc[i, event_time_col]
        t_end = events.loc[i + 1, event_time_col]
        tauc = events.loc[i, tauc_col]

        # subset shear stress between events
        mask = (shear_stress[shear_time_col] > t_start) & \
            (shear_stress[shear_time_col] < t_end)
        shear_between = shear_stress.loc[mask, shear_col]

        # exclude values above critical shear stress
        shear_below_tauc = shear_between[shear_between <= tauc]

        delta_t_hours = (t_end - t_start).total_seconds() / 3600
        mean_tau_below = shear_below_tauc.mean()

        results.append({
            "event_start": t_start,
            "event_end": t_end,
            "tauc": tauc,
            "time_between_events_hours": delta_t_hours,
            "mean_shear_below_tauc": mean_tau_below,
            "n_timesteps_total": shear_between.size,
            "n_timesteps_below_tauc": shear_below_tauc.size
        })

    return pd.DataFrame(results)


## No history dependent tau

### 1. Interpolated Rising D50

In [ ]:
events = "field_data/1_interpolated_rising_d50.csv"
antecedent_1 = antecedent_average(events_csv=events,shear_stress=shear_stress)

print(antecedent_1)

          event_start           event_end  time_between_events_hours  \
0 2022-08-03 14:55:00 2022-08-08 14:25:00                 119.500000   
1 2022-08-08 14:25:00 2023-04-28 02:46:00                6300.350000   
2 2023-04-28 02:46:00 2023-07-29 14:38:00                2219.866667   
3 2023-07-29 14:38:00 2023-09-14 22:31:00                1135.883333   
4 2023-09-14 22:31:00                 NaT                        NaN   
5                 NaT                 NaT                        NaN   

   mean_shear_between_events  n_timesteps  
0                   0.002619          478  
1                   0.034736         9363  
2                   0.033686         8879  
3                   0.000805         4544  
4                        NaN            0  
5                        NaN            0  
